In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, upper, trim
from delta import configure_spark_with_delta_pip

# ===================================================
# 1. INICIAR SESSÃO
# ===================================================
warehouse_location = "hdfs://hdfs-nn:9000/warehouse"

builder = SparkSession.builder \
    .appName("Gold_Fact_Info_Pais_Clean") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()
print("✅ Sessão iniciada.")

# ===================================================
# 2. DEFINIR O "DICIONÁRIO" DE PAÍSES (Em Memória)
# ===================================================
# Isto substitui a tabela 'silver.dim_country_codes' que apagaste.
# Serve para traduzir o nome que vem no GDP para o código que a Dimensão entende.
mapa_paises_data = [
    ("US", "United States"), ("GB", "United Kingdom"), ("IN", "India"),
    ("FR", "France"), ("CA", "Canada"), ("DE", "Germany"),
    ("JP", "Japan"), ("ES", "Spain"), ("IT", "Italy"), ("BR", "Brazil"),
    ("MX", "Mexico"), ("AU", "Australia"), ("KR", "South Korea"),
    ("CN", "China"), ("PT", "Portugal"), ("NL", "Netherlands"),
    ("SE", "Sweden"), ("NO", "Norway"), ("DK", "Denmark"),
    ("RU", "Russia"), ("IE", "Ireland"), ("BE", "Belgium"),
    ("ZA", "South Africa"), ("NZ", "New Zealand"), ("AR", "Argentina"),
    ("CH", "Switzerland"), ("TR", "Turkey"), ("ID", "Indonesia")
]
# Criar DataFrame temporário
df_map_temp = spark.createDataFrame(mapa_paises_data, ["alpha2", "gdp_name"])
df_map_temp.createOrReplaceTempView("mapa_temp")

print("✅ Dicionário de países criado em memória.")

# ===================================================
# 3. LER DADOS
# ===================================================
path_dim_paises = f"{warehouse_location}/gold.db/dim_paises"
path_dim_calendario = f"{warehouse_location}/gold.db/dim_calendario"
path_silver_gdp = f"{warehouse_location}/silver.db/gdp_data"

try:
    print("🔄 A ler tabelas...")
    spark.read.format("delta").load(path_dim_paises).createOrReplaceTempView("dim_paises")
    spark.read.format("delta").load(path_dim_calendario).createOrReplaceTempView("dim_calendario")
    spark.read.format("delta").load(path_silver_gdp).createOrReplaceTempView("silver_gdp")

    # ===================================================
    # 4. CRIAR FACT TABLE (Usando o mapa temporário)
    # ===================================================
    print("🏗️ A cruzar dados...")
    
    # A Lógica agora é:
    # GDP (Nome: 'United States') -> Mapa Temp (Nome -> Sigla 'US') -> Dim País (Sigla -> ID 'P1')
    
    df_fact = spark.sql("""
        SELECT
            c.id_calendario,
            p.id_pais,
            g.GDP_Per_Capita as gdp_per_capita,
            g.Year as ano_original
        FROM silver_gdp g
        
        -- 1. Ligar ao Calendário
        JOIN dim_calendario c ON g.Year = c.ano
        
        -- 2. TRUQUE: Ligar ao Mapa Temporário para traduzir o nome
        JOIN mapa_temp m ON g.Country_Name = m.gdp_name
        
        -- 3. Ligar à Dimensão País oficial pela Sigla (que é segura)
        JOIN dim_paises p ON m.alpha2 = p.sigla
    """)

    # ===================================================
    # 5. GRAVAR
    # ===================================================
    fact_path = f"{warehouse_location}/gold.db/fact_info_pais"
    print(f"💾 A gravar em: {fact_path}")

    df_fact.write \
        .format("delta") \
        .mode("overwrite") \
        .partitionBy("ano_original") \
        .option("overwriteSchema", "true") \
        .save(fact_path)

    spark.sql(f"CREATE DATABASE IF NOT EXISTS gold LOCATION '{warehouse_location}/gold.db'")
    spark.sql(f"CREATE TABLE IF NOT EXISTS gold.fact_info_pais USING DELTA LOCATION '{fact_path}'")

    print("🎉 SUCESSO! Tabela 'fact_info_pais' criada sem ficheiros extra.")
    spark.sql("SELECT * FROM gold.fact_info_pais LIMIT 5").show()

except Exception as e:
    print(f"❌ Erro: {e}")

✅ Sessão iniciada.
✅ Dicionário de países criado em memória.
🔄 A ler tabelas...
🏗️ A cruzar dados...
💾 A gravar em: hdfs://hdfs-nn:9000/warehouse/gold.db/fact_info_pais
🎉 SUCESSO! Tabela 'fact_info_pais' criada sem ficheiros extra.
+-------------+-------+--------------+------------+
|id_calendario|id_pais|gdp_per_capita|ano_original|
+-------------+-------+--------------+------------+
|        C2001|  P1008|     37133.242|        2001|
|        C2001|   P554|     2206.4482|        2001|
|        C2001|   P936|      27807.17|        2001|
|        C2001|   P343|      27490.05|        2001|
|        C2001|   P153|     30230.803|        2001|
+-------------+-------+--------------+------------+

